In [2]:
from osgeo import gdal
import os

In [1]:
def list_raster_bands(raster_path):
    """
    Lists the bands and their descriptions for a multiband raster.

    Parameters:
        raster_path (str): Path to the raster file.

    Returns:
        list of tuples: Each tuple is (band_number, band_description)
    """
    dataset = gdal.Open(raster_path)
    if not dataset:
        raise Exception(f"Could not open {raster_path}")

    num_bands = dataset.RasterCount
    bands_info = []

    for band_index in range(1, num_bands + 1):
        band = dataset.GetRasterBand(band_index)
        desc = band.GetDescription()
        bands_info.append((band_index, desc))

    dataset = None
    return bands_info

def export_raster_bands(raster_path, output_folder):
    """
    Exports each band of a multiband raster as a single-band GeoTIFF,
    using the band description as the filename if available.

    Parameters:
        raster_path (str): Path to the raster file.
        output_folder (str): Folder where output files will be saved.
    """
    dataset = gdal.Open(raster_path)
    if not dataset:
        raise Exception(f"Could not open {raster_path}")

    num_bands = dataset.RasterCount

    for band_index in range(1, num_bands + 1):
        band = dataset.GetRasterBand(band_index)
        desc = band.GetDescription().strip() or f"band_{band_index}"

        # Make a safe filename
        safe_desc = desc.replace(" ", "_")
        output_file = os.path.join(output_folder, f"{safe_desc}.tif")

        # Read band data
        band_data = band.ReadAsArray()

        # Create output dataset with Deflate compression & tiling
        driver = gdal.GetDriverByName('GTiff')
        out_ds = driver.Create(
            output_file,
            dataset.RasterXSize,
            dataset.RasterYSize,
            1,
            band.DataType,
            options=[
                'COMPRESS=DEFLATE',
                'TILED=YES'
            ]
        )

        # Copy georeferencing
        out_ds.SetGeoTransform(dataset.GetGeoTransform())
        out_ds.SetProjection(dataset.GetProjection())

        # Write the data
        out_band = out_ds.GetRasterBand(1)
        out_band.WriteArray(band_data)

        nodata = band.GetNoDataValue()
        if nodata is not None:
            out_band.SetNoDataValue(nodata)

        out_band.FlushCache()
        out_ds.FlushCache()

        print(f"Saved: {output_file}")

    dataset = None
    print("All bands exported.")


In [3]:
raster_file = r"C:\Users\admin\Downloads\rast_gdpTot_1990_2020_30arcsec.tif"

# Output folder
output_dir = r"C:\Users\admin\Downloads\output_folder"

bands = list_raster_bands(raster_file)
for band_number, description in bands:
    print(f"Band {band_number}: '{description}'")


export_raster_bands(raster_file, output_dir)

Band 1: 'gdp_1990'
Band 2: 'gdp_1995'
Band 3: 'gdp_2000'
Band 4: 'gdp_2005'
Band 5: 'gdp_2010'
Band 6: 'gdp_2015'
Band 7: 'gdp_2020'
Saved: C:\Users\admin\Downloads\output_folder\gdp_1990.tif
Saved: C:\Users\admin\Downloads\output_folder\gdp_1995.tif
Saved: C:\Users\admin\Downloads\output_folder\gdp_2000.tif
Saved: C:\Users\admin\Downloads\output_folder\gdp_2005.tif
Saved: C:\Users\admin\Downloads\output_folder\gdp_2010.tif
Saved: C:\Users\admin\Downloads\output_folder\gdp_2015.tif
Saved: C:\Users\admin\Downloads\output_folder\gdp_2020.tif
All bands exported.
